# Ensemble Counterfactual Diagnostics for z-Only CSE

This notebook investigates **why the interventions selected by the existing CSE algorithm work when only $z$ is directly perturbed**. It does not modify the CSE implementation or use the future uncontrolled nature-run transition to decide when to intervene.

The original CSE run is first reproduced with the same configuration and random seed. Counterfactual forecasts are then launched only at cycles that the original ensemble forecast already caused CSE to control. Each analysis ensemble is propagated under three diagnostic alternatives: no intervention, the stored CSE-selected $u_z$, and the opposite $-u_z$. These branches do not feed back into CSE.


In [ ]:
import importlib
import pickle

import numpy as np

import project_paths as project_paths
import class_cse_qiwen as cse_qiwen
import class_cse_counterfactual_diagnostics as counterfactual

project_paths = importlib.reload(project_paths)
cse_qiwen = importlib.reload(cse_qiwen)
counterfactual = importlib.reload(counterfactual)

from class_da_sys import da_system

EXPERIMENT_NAME = 'z_control_nature_qiwenevents'
EXPERIMENT_ROOT, RESULTS_DIR, FIGURES_DIR = project_paths.cse_experiment_paths(
    EXPERIMENT_NAME
)
DIAGNOSTICS_DIR = EXPERIMENT_ROOT / 'diagnostics'
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_PATH = (
    RESULTS_DIR / 'cse_z_only_nature_run_event_1_D0p05_qiwenevents.pkl'
)
OUTPUT_PATH = (
    DIAGNOSTICS_DIR
    / 'cse_z_only_ensemble_counterfactuals_event_1_D0p05.pkl'
)

print('Reference result:', REFERENCE_PATH)
print('Counterfactual output:', OUTPUT_PATH)


## Reproduce the Original CSE Run

The compact reference file contains the controlled and uncontrolled nature trajectories and the applied interventions, but not the complete forecast-ensemble history. Therefore, the unchanged authoritative `run_single_cse` function is executed again with the saved configuration. The resulting controlled trajectory, branch history, and intervention array must match the reference before any counterfactual is accepted.

The saved list of regime crossings is used only to reconstruct the same event index that initialized the original experiment. No crossing from the uncontrolled trajectory is used to select the diagnostic cycles or counterfactual interventions.


In [ ]:
with open(REFERENCE_PATH, 'rb') as stream:
    reference = pickle.load(stream)

summary = dict(reference['summary'])
das = da_system().load(project_paths.ANALYSIS_DATA_DIR / 'x_analysis_init.pkl')
das.multiplicative_inflation = 1.04
x_nature = das.getStateVector().getTrajectory()

with open(project_paths.OSSE_DATA_DIR / 'osse_Xa_history.pkl', 'rb') as stream:
    osse_Xa_history = pickle.load(stream)
with open(project_paths.OSSE_DATA_DIR / 'osse_summary.pkl', 'rb') as stream:
    osse_summary = pickle.load(stream)
with open(RESULTS_DIR / 'change_region_time_qiwen_qiwenevents.pkl', 'rb') as stream:
    change_region_time = pickle.load(stream)

# Exact Qiwen event filtering used by the original notebook.
t_start_series = [
    crossing
    for index, crossing in enumerate(change_region_time)
    if crossing >= 500 and crossing - change_region_time[index - 1] > 385
][:100]

print('Event index:', summary['run_cse_event_index'])
print('Selected event time:', t_start_series[summary['run_cse_event_index']])
print('Saved t_start:', summary['t_start'])
assert t_start_series[summary['run_cse_event_index']] == summary['t_start']


In [ ]:
single_cse_result = cse_qiwen.run_single_cse(
    event_index=summary['run_cse_event_index'],
    D_run=summary['D'],
    seed=summary['seed'],
    das=das,
    x_nature=x_nature,
    t_start_series=t_start_series,
    osse_Xa_history=osse_Xa_history,
    osse_summary=osse_summary,
    experiment_step=summary['requested_num_cycles'],
    position=summary['position'],
    Ta=summary['Ta'],
    fcst_dt=summary['fcst_dt'],
    path_figures=None,
    total_events=len(t_start_series),
    regime_reference=summary['regime_reference'],
    candidate_pool=summary['candidate_pool'],
    control_indices=summary['control_indices'],
    T_0=summary['T_0'],
    multiple_number=summary['multiple_number'],
    cle_target_output_dir=None,
)


In [ ]:
Ta = int(summary['Ta'])
n_steps = int(single_cse_result['n_completed']) * Ta
reproduced_control = np.asarray(single_cse_result['evo_history'])[:, :n_steps + 1]
reproduced_interventions = np.zeros_like(reproduced_control)
for cycle, item in enumerate(single_cse_result['intervention_history']):
    start = cycle * Ta
    reproduced_interventions[:, start:start + Ta] = np.asarray(
        item['perturbations']
    ).T

control_error = float(np.max(np.abs(
    reproduced_control - np.asarray(reference['nature_control'])
)))
intervention_error = float(np.max(np.abs(
    reproduced_interventions - np.asarray(reference['interventions'])
)))
branches_match = (
    list(single_cse_result['branch_history']) == list(reference['branch_history'])
)

assert control_error <= 1e-12
assert intervention_error <= 1e-12
assert branches_match

print('Maximum controlled-trajectory error:', control_error)
print('Maximum intervention error:', intervention_error)
print('Branch histories match:', branches_match)
print('Completed cycles:', single_cse_result['n_completed'])
print('Branch counts:', single_cse_result['branch_counts'])


## Counterfactual Ensemble Forecasts

For every cycle already controlled by CSE, all ensemble members start from the same saved analysis ensemble. The diagnostic branches differ only in the intervention packet:

\[
u_z^{\mathrm{free}}=0,\qquad
u_z^{\mathrm{CSE}}=u_z,\qquad
u_z^{\mathrm{opposite}}=-u_z.
\]

The increments retain the original discrete alignment: row zero is zero and rows $1, ..., T_a-1$ are applied after their corresponding free RK4 advances. Risk metrics use the original 301-step CSE forecast horizon. For Figure 6 only, the same post-decision branches are continued freely to step 400 so that their longer-term separation remains visible.


In [ ]:
diagnostics = counterfactual.build_counterfactual_diagnostics(
    single_cse_result,
    position=summary['position'],
    Ta=summary['Ta'],
    dt=summary['fcst_dt'],
    representative_position=400,
)
counterfactual.save_counterfactual_diagnostics(diagnostics, OUTPUT_PATH)

print('Controlled cycles analysed:', len(diagnostics['controlled_cycles']))
print('Risk-metric horizon:', diagnostics['position'])
print('Representative-figure horizon:', diagnostics['representative_position'])
print('Representative cycles:')
for cycle, record in diagnostics['representative_cycles'].items():
    print(f"  {cycle}: {record['selection_label']}")
print('Maximum free-forecast replay error:',
      diagnostics['validation']['maximum_free_forecast_abs_error'])
print('All interventions are z-only:',
      diagnostics['validation']['all_interventions_z_only'])
print('All crossing counts reproduce CSE:',
      diagnostics['validation']['all_free_crossing_counts_match_cse'])
print('Saved:', OUTPUT_PATH)


In [ ]:
aggregate = diagnostics['aggregate']
print('Mean crossing fraction by branch:')
for branch, value in aggregate['mean_crossing_fraction'].items():
    print(f'  {branch:>8}: {value:.6f}')

print('Fraction of cycles where CSE has lower risk than no intervention:',
      aggregate['cse_lower_risk_than_free_fraction'])
print('Fraction of cycles where CSE has higher risk than no intervention:',
      aggregate['cse_higher_risk_than_free_fraction'])
print('Fraction of cycles where CSE has a higher margin than no intervention:',
      aggregate['cse_higher_margin_than_free_fraction'])
print('Fraction of cycles where CSE has lower risk than the opposite action:',
      aggregate['cse_lower_risk_than_opposite_fraction'])
print('Fraction of cycles where CSE has a higher margin than the opposite action:',
      aggregate['cse_higher_margin_than_opposite_fraction'])


## Interpretation Boundary

These are **post-decision causal diagnostics**, not a replacement controller. The uncontrolled nature trajectory is not used to choose a cycle, intervention sign, or intervention onset. It remains available only for the separate, posterior verification performed in the mechanism notebook.

Because each diagnostic branch isolates one intervention packet and does not alter later CSE decisions, the results measure the local forecast effect of the selected $u_z$. The complete controlled nature run still represents the cumulative closed-loop result of repeatedly forecasting, intervening, observing, and assimilating.
